```{contents}
:local:
:depth: 2
```

# Problems: Model Validation

:::{admonition} Before you start
:class: tip

This problem set accompanies {doc}`Model Validation </2-regression/Topic2.2-Model_Validation>`. It is worth **100 points**:

- **Part A — Skill Checks (30 pts)** — short answers, auto-graded, resubmit as often as
  you like until they pass.
- **Part B — Visualization (35 pts)** — plots plus written interpretation, peer graded.
- **Part C — Open Ended (35 pts)** — one synthesis problem, peer graded.

The parts build on each other: Part A works out the syntax you need for Part B, and
Part B produces the evidence you argue from in Part C. Do them in order.
:::

## Setup

Henry's law constant $H$ describes how much of a gas dissolves in water at equilibrium.
Its temperature dependence is set by the enthalpy of solvation $\Delta H_\text{sol}$
through the van 't Hoff relation. A **linear free-energy relationship** predicts that
species which dissolve more exothermically should also be more soluble — that is, that
$\ln H$ should fall roughly linearly with $\Delta H_\text{sol}$.

You will test that claim on Sander's compilation of Henry's law constants (version 5.0.0),
which gathers values from the published literature for thousands of species. Each row is
**one species as reported by one literature reference**:

| column | meaning |
|---|---|
| `species`, `formula`, `casrn` | chemical identity |
| `ref` | which literature reference reported this value |
| `htype` | how the value was obtained — `M` measured, `L` literature review, `V` from VLE data, `T` from thermodynamic data, `Q` estimated by a QSAR |
| `H_mol_m3_Pa`, `lnH` | Henry's law solubility constant at 298.15 K, and its log |
| `mindHR_K`, `dHsol_kJ_mol` | van 't Hoff temperature coefficient, and $\Delta H_\text{sol}$ |

The important structural feature: **many species appear more than once**, because several
groups measured them independently. That gives you something rare — a direct measurement
of how much independent laboratories disagree.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
try:
    plt.style.use('../settings/plot_style.mplstyle')   # available inside the book
except OSError:
    pass                                              # downloaded notebook: use defaults

df = pd.read_csv('data/henry_law.csv')
print(f"{len(df)} rows, {df.species.nunique()} species, {df.ref.nunique()} references")
df.head()

In [ ]:
counts = df.groupby('species').size()
print(f"species reported by >= 3 independent references: {(counts >= 3).sum()}")
print(df.htype.value_counts().to_string())

Run this once to load the autograder. It will not run inside the book — use the
downloaded notebook.

In [ ]:
import otter
grader = otter.Notebook()

---

## Part A — Skill Checks (30 pts)

Three questions, 10 points each. Throughout, the model is an ordinary least-squares fit
of `lnH` (target) on the single feature `dHsol_kJ_mol`, using **all 4,344 rows**.

### A1. Fit the relationship (10 pts)

:::{exercise}
:label: pr-reg-henry-lfer

Fit `sklearn.linear_model.LinearRegression` with `dHsol_kJ_mol` as the only feature and
`lnH` as the target, using the whole dataset. Report the $r^2$ **on the same data you fit
on**.

Assign it to `r2_full`.
:::

In [ ]:
# YOUR CODE HERE
r2_full = ...

In [ ]:
grader.check("q1")

### A2. Validate it honestly (10 pts)

:::{exercise}
:label: pr-reg-henry-cv

The A1 number is optimistic — it scores the model on the data used to fit it. Redo the
evaluation with 5-fold cross-validation, using exactly

```
KFold(n_splits=5, shuffle=True, random_state=0)
```

Report the **mean** of the five fold $r^2$ values and assign it to `r2_cv`.

The seed matters. Do not change it — part of the point is that a shuffled split is a
random quantity, and a reported CV score is meaningless unless the split is reproducible.
:::

In [ ]:
# YOUR CODE HERE
r2_cv = ...

In [ ]:
grader.check("q2")

### A3. Measure the noise floor (10 pts)

:::{exercise}
:label: pr-reg-henry-floor

When several references report the same species, they disagree. That disagreement is
measurement noise, and **no model can predict it** — it sets a floor on the error any
model can achieve.

Estimate it. Restrict to species with **at least 3** independent reports, compute the
sample standard deviation of `lnH` within each species, and pool them:

$$
\sigma_\text{floor} \;=\; \sqrt{\frac{\sum_s (n_s - 1)\,s_s^2}{\sum_s (n_s - 1)}}
$$

where $n_s$ and $s_s$ are the count and standard deviation for species $s$. Use the
sample standard deviation (`ddof=1`, which is pandas' default).

Assign the result to `sigma_floor`.
:::

In [ ]:
# YOUR CODE HERE
sigma_floor = ...

In [ ]:
grader.check("q3")

---

## Part B — Visualization (35 pts)

:::{exercise}
:label: pr-reg-henry-diag

Build a three-panel figure using the model and quantities from Part A.

1. **Predicted vs. actual.** Use `cross_val_predict` with the same `KFold` from A2 to get
   an out-of-fold prediction for every row, and scatter predicted against actual `lnH`.
   Draw the 1:1 line. Color the points by whether `htype` is `'Q'` (a QSAR estimate) or
   anything else (an experimental or literature value).
2. **How stable is the CV score?** Recompute the mean CV $r^2$ for `n_splits` in
   $\{2, 5, 10, 20\}$, each for at least 10 different `random_state` values, and show the
   spread — a boxplot per `n_splits`.
3. **The noise floor.** Histogram the per-species standard deviation of `lnH` for the 362
   species with ≥3 reports, and mark $\sigma_\text{floor}$ from A3 with a vertical line.

Then, in **3–5 sentences**: panel 1 shows the `'Q'` points hugging the 1:1 line more
tightly than the experimental ones. Explain why that is expected, and say what it implies
about using $r^2$ on this dataset as evidence that the linear free-energy relationship is
physically real.

Label all axes with units.
:::

In [ ]:
# YOUR CODE HERE
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

---

## Part C — Open Ended (35 pts)

:::{exercise}
:label: pr-reg-henry-headroom

You now have two numbers that measure very different things: the cross-validated error of
your model, and $\sigma_\text{floor}$, the error that measurement disagreement alone would
produce even for a perfect model.

Decide whether this model is worth improving, and defend it with evidence.

Your answer should include:

1. The cross-validated **RMSE** in ln units (not $r^2$), compared directly against
   $\sigma_\text{floor}$ from A3.
2. An estimate of the best $r^2$ any model could reach on this dataset if the only
   remaining error were measurement noise. State your assumptions.
3. At least one concrete attempt to close the gap — a second feature, a nonlinear model
   from {doc}`Non-parametric Models </2-regression/Topic2.1-Non-parametric_Models>`, restricting to a subset of `htype`, or
   anything else you can justify — evaluated with the same cross-validation procedure.
4. A short written argument (**one paragraph**) about whether the effort is warranted,
   citing your own numbers.

Be careful with step 3: at least one obvious-looking move makes the reported score *worse*
for a reason that is a feature of the data rather than a failure of the model. If you hit
it, explain it.

There is more than one defensible conclusion. You are graded on the reasoning and the
evidence.
:::

In [ ]:
# YOUR CODE HERE

---

## Summary

- Part A produced three numbers that answer three different questions: how well the model
  fits the data it saw (0.602), how well it predicts data it did not (0.594), and how much
  disagreement exists between independent measurements of the same quantity (0.53 ln units).
- Part B showed that the CV estimate is stable across fold counts and seeds, and that part
  of the apparent fit quality comes from QSAR-estimated rows that are model output rather
  than independent observations.
- Part C set the model's error against the measurement floor. A cross-validated score is
  only interpretable once you know what error would remain even if the model were perfect.

## Additional Reading

1. R. Sander, [Compilation of Henry's law constants (version 5.0.0) for water as solvent](https://doi.org/10.5194/acp-23-10901-2023),
   *Atmos. Chem. Phys.* **23**, 10901–12440 (2023).
2. [`sklearn.model_selection.cross_val_predict`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_predict.html)
   — and the note in its documentation on why its output is not appropriate for computing
   a generalization metric in general.